# Why Deep Learning?

Imagine building a system that reads handwritten digits on envelopes. A computer receives a grid of pixel brightness values, but the same digit can move, tilt, or be written in different styles. Knowing how to fit a classifier is only part of the problem. We also need a useful way to describe what is in the image.

A **feature** is an input quantity used by a model. We might design features such as the number of loops or the direction of a stroke. That sounds helpful until we try to write reliable rules for extracting them from every style of handwriting.

Deep learning addresses this difficulty by letting a model learn intermediate descriptions of its inputs while learning a task. To understand why that helps, we will first examine a much smaller problem whose geometry we can see and calculate. This lesson assumes introductory machine learning, but no prior knowledge of neural networks.

The [code companion](01_Why_Deep_Learning.ipynb) implements the comparison described here.

## A prediction can be difficult because of its inputs

Suppose a plotting tool must color points according to whether they lie inside or outside a circle of radius 1 centered at the origin. Each point has a horizontal coordinate $x_1$ and a vertical coordinate $x_2$. Let the target label $y$ be 0 inside or on the circle and 1 outside it.

A familiar starting model is logistic regression. With these two raw coordinates, it forms a score

$$z=w_1x_1+w_2x_2+b.$$

$z$ is the score, $w_1$ and $w_2$ are learned multipliers called **weights**, and $b$ is a learned offset called a **bias**. Logistic regression converts the score into a class-1 probability estimate. Using the usual probability cutoff of 0.5, its **decision boundary**—where the predicted class changes—is the line $z=0$.

That line cannot enclose a circular region. Points outside the circle surround points inside it, whereas a straight line divides the plane into two sides. More training cannot change this restriction on the model's boundary.

The problem is not that every traditional machine learning method is inadequate. Other methods, including decision trees, can describe nonlinear boundaries. We have identified a specific mismatch between this model and these input features.

## A useful representation can make the decision simple

We know something about circles: distance from the center determines membership. Instead of describing each point with two coordinates, calculate its squared distance:

$$r^2=x_1^2+x_2^2.$$

$r$ is the distance from the origin, and $r^2$ is its square. Squaring each coordinate and adding gives:

| Point | $x_1$ | $x_2$ | Calculation of $r^2$ | Label $y$ |
| --- | ---: | ---: | --- | ---: |
| A | 0.3 | 0.4 | $0.09+0.16=0.25$ | 0 |
| B | 0.8 | 0.8 | $0.64+0.64=1.28$ | 1 |
| C | -0.8 | -0.8 | $0.64+0.64=1.28$ | 1 |
| D | 0.0 | 0.0 | $0+0=0$ | 0 |

For A, the squared distance 0.25 is below 1, so the point is inside. For B and C, it is above 1, so they are outside. A square root is unnecessary: distance exceeds 1 exactly when its square exceeds 1.

Now a simple score $z=r^2-1$ is sufficient. Predict outside when $z>0$, otherwise inside. The boundary is just one cutoff in the new feature, even though it is curved in the original coordinate plane.

A **representation** is a numerical description of data that exposes information useful for a task. We changed the representation from coordinates to squared distance. Designing such quantities ourselves is called **feature engineering**.

```text
Coordinates (x1, x2)  →  squared distance r²  →  compare with 1
  circular boundary        one useful value       simple decision
```

For this known rule, geometry is enough; no model needs to be trained. We will also fit logistic regression on squared distance in the companion, so we can compare a designed representation with a learned one. Its geometric knowledge is an explicit advantage we gave that model.

## What if we cannot easily design the representation?

The circle gave us an obvious transformation. Handwriting does not provide such a short recipe. A useful feature may need to combine many pixels while ignoring irrelevant changes in position, thickness, or background.

A **neural network** is a model composed of connected computation stages with adjustable numerical values, or **parameters**. A **layer** groups computations that produce a new set of values from the preceding input. An intermediate, or **hidden**, layer supplies values to another layer instead of directly producing the final answer.

Each stage combines incoming quantities and can apply a **nonlinear transformation**, one whose complete response cannot be described by a single weighted sum plus an offset. For example, ReLU, short for rectified linear unit, keeps positive scores and replaces negative scores with zero. Its response changes at zero. Combining many such responses lets a network form boundaries that a single straight-line classifier cannot express.

Instead of fixing the intermediate features ourselves, we fit their parameters together with the final prediction rule. This is **representation learning**:

```text
Designed features: input → transformation chosen by a person → fitted predictor
Learned features:  input → adjustable transformations         → fitted predictor
                          └──── trained together ────────────────────┘
```

Training compares predictions with known targets using a **loss**, a numerical measure of mismatch, then adjusts parameters to reduce it. Because the output depends on the intermediate transformations, the same prediction error can guide changes in those transformations too. This joint fitting is called **end-to-end learning**.

The learned values need not match features we would name ourselves. For the circle, a network may approximate the boundary using several intermediate responses without computing squared distance exactly. Useful information can be distributed across units rather than stored in one easily interpreted feature.

## What makes it deep?

**Deep learning** is a branch of machine learning that uses neural networks with multiple learned processing layers to build representations in stages. **Depth** describes the sequence of layers; **width** describes the number of units within a layer. Neither refers to the number of training examples or training repetitions.

A later layer can combine patterns detected earlier. In an image task, an early representation might respond to local brightness changes, while later combinations become useful for recognizing larger patterns. This is an intuition for why composition can help, not a guarantee that every layer will have a neat human-readable meaning.

The nonlinear steps matter. Repeated weighted sums and offsets without nonlinearities can be combined into one weighted sum and offset, so merely stacking them does not escape the straight-boundary limitation. With nonlinearities, intermediate responses can be reused and combined into a more flexible function.

The companion uses two hidden layers with 16 units each. It is deliberately small, and this task does not require that particular depth: one hidden layer can already approximate a circular boundary. The experiment illustrates the value of learning transformations, not a benefit unique to very deep models.

## How the companion tests the idea

We generate balanced inside and outside examples locally and compare three models on identical data:

| Model | Information it receives | What is fitted |
| --- | --- | --- |
| Logistic regression on coordinates | Two raw coordinates | A straight-boundary prediction rule |
| Logistic regression on squared distance | A feature designed using geometry | A prediction rule on that feature |
| Small neural network | Two raw coordinates | Intermediate transformations and a prediction rule |

The **training set** fits parameters. A separate **validation set** selects a saved parameter state, called a **checkpoint**. The **test set** is reserved for a final comparison on unseen points. Using test information to choose preprocessing or model settings would be **data leakage**, making the result overly optimistic. The inputs already share a fixed range, so no fitted scaling is necessary here.

**Accuracy** is the fraction of labels predicted correctly. We report it alongside boundary plots, which reveal what the models actually do. Training performance alone is insufficient: a model can **overfit**, adapting to training examples in a way that fails on new ones.

Interpret the results as evidence about these features and this task. A successful network shows that it can learn useful transformations without receiving our distance formula. A successful distance-based model shows why feature engineering and simple methods remain valuable. A few errors on one small test set do not establish a general ranking of algorithms.

## When the extra machinery is worthwhile

The strongest reason to consider deep learning is that useful input structure is difficult to specify manually. Images, audio, and text can contain many interacting measurements, and learning representations can make these tasks more manageable. Deep learning is also used in generative systems that produce new data, but a classifier like our example does not acquire that ability merely by having hidden layers.

The flexibility comes with costs. Training larger networks can require substantial data, computation, and experimentation. A model may learn an accidental shortcut, such as an image background correlated with a label, rather than the intended pattern. Its hidden representations can also be difficult to interpret. Human choices about data, objectives, architecture, and evaluation still shape what it learns.

When a known rule solves the problem, use it. When a small table already contains meaningful features, establish simple baselines before investing in a larger model. Check performance on examples that reflect the actual task, along with runtime and interpretability needs. A neural network is worthwhile when those comparisons justify its added complexity—not simply because it has more layers.